In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(ROOT))

In [3]:
import shutil
import pandas as pd
from pathlib import Path
from src.utils.helpers import totalizador, planilha_lancamento

origem_lanc = Path(r"C:\Users\manja\OneDrive\Documentos\AutoExtrato\data\Lancamentos_Contabeis.xlsm")
destino_lanc = Path(r"G:\.shortcut-targets-by-id\1nboB01tRteAh3IDImIuSVc8ksH6cRrEH\PROJ EXT\156_CEMAF OPE\Extratos\26\04")
pasta_ext = Path(r"G:\.shortcut-targets-by-id\1nboB01tRteAh3IDImIuSVc8ksH6cRrEH\PROJ EXT\156_CEMAF OPE\Extratos\26\04")
arquivo = Path(r"G:\.shortcut-targets-by-id\1nboB01tRteAh3IDImIuSVc8ksH6cRrEH\PROJ EXT\156_CEMAF OPE\Extratos\26\04\0426_FECFIN_CEMAF OPE.xlsx")

with pd.ExcelFile(arquivo) as xls:
    abas = xls.sheet_names

    if "CEMAF" in arquivo.stem:
        bancos = ["UNICRED", "SICOOB", "SICREDI", "CAIXA"]
        extratos = [aba for aba in abas for banco in bancos if banco in aba]

        arquivo_ext = pasta_ext / f"[EXT] {arquivo.stem}.xlsx"
        with pd.ExcelWriter(arquivo_ext, engine="openpyxl") as writer:
            for extrato in extratos:
                df = pd.read_excel(xls, sheet_name=extrato)

                if "UNICRED" in extrato:
                    df.columns = df.iloc[5]
                    df = df[6:].reset_index(drop=True)

                    df["DESCRIÇÃO"] = df.apply(
                        lambda x: f'{x["HISTÓRICO"]} {x["OBS"]} {x["OBS P/ INTERNAS"]} {x["TIPO"]} {x["Nº DOC"]}',
                        axis=1
                    )

                    df["DESCRIÇÃO"] = df["DESCRIÇÃO"].str.replace("nan", "", regex=False).str.upper()
                    df = df[["DATA", "DESCRIÇÃO", "ENTRADA", "SAIDA", "SALDO"]]

                    df["ENTRADA"] = pd.to_numeric(df["ENTRADA"], errors="coerce").fillna(0)
                    df["SAIDA"] = pd.to_numeric(df["SAIDA"], errors="coerce").fillna(0)

                    df["VALOR"] = df["ENTRADA"].where(df["ENTRADA"] != 0, df["SAIDA"] * -1)
                    df = df.loc[~df["DESCRIÇÃO"].astype(str).str.upper().str.contains("SALDO", na=False)]

                    df["TIPO"] = df["VALOR"].apply(lambda valor: "C" if valor > 0 else "D")
                    df = df[["DATA", "DESCRIÇÃO", "VALOR", "TIPO"]]
                    df["VALOR"] = df["VALOR"].abs()

                elif "SICOOB" in extrato:
                    df.columns = df.iloc[5]
                    df = df[6:].reset_index(drop=True)

                    df["DESCRIÇÃO"] = df.apply(
                        lambda x: f'{x["HISTÓRICO"]} {x["OBS P/ CONT"]} {x["OBS P/ INTERNAS"]} {x["TIPO"]} {x["Nº DOC"]}',
                        axis=1
                    )

                    df["DESCRIÇÃO"] = df["DESCRIÇÃO"].str.replace("nan", "", regex=False).str.upper()
                    df = df[["DATA", "DESCRIÇÃO", "ENTRADA", "SAIDA", "SALDO"]]
                    df = df.dropna(subset=["DATA"])

                    df["ENTRADA"] = pd.to_numeric(df["ENTRADA"], errors="coerce").fillna(0)
                    df["SAIDA"] = pd.to_numeric(df["SAIDA"], errors="coerce").fillna(0)

                    df["VALOR"] = df["ENTRADA"].where(df["ENTRADA"] != 0, df["SAIDA"] * -1)
                    df = df.loc[~df["DESCRIÇÃO"].astype(str).str.upper().str.contains("SALDO", na=False)]

                    df["TIPO"] = df["VALOR"].apply(lambda valor: "C" if valor > 0 else "D")
                    df = df[["DATA", "DESCRIÇÃO", "VALOR", "TIPO"]]
                    df["VALOR"] = df["VALOR"].abs()

                elif "SICREDI" in extrato:
                    df.columns = df.iloc[6]
                    df = df[7:].reset_index(drop=True)

                    df["DESCRIÇÃO"] = df.apply(
                        lambda x: f'{x["HISTÓRICO"]} {x["OBS"]} {x["TIPO"]} {x["Nº DOC"]}',
                        axis=1
                    )

                    df["DESCRIÇÃO"] = df["DESCRIÇÃO"].str.replace("nan", "", regex=False).str.upper()
                    df = df[["DATA", "DESCRIÇÃO", "ENTRADA", "SAIDA", "SALDO"]]

                    df["ENTRADA"] = pd.to_numeric(df["ENTRADA"], errors="coerce").fillna(0)
                    df["SAIDA"] = pd.to_numeric(df["SAIDA"], errors="coerce").fillna(0)

                    df["VALOR"] = df["ENTRADA"].where(df["ENTRADA"] != 0, df["SAIDA"] * -1)
                    df = df.loc[~df["DESCRIÇÃO"].astype(str).str.upper().str.contains("SALDO", na=False)]

                    df["TIPO"] = df["VALOR"].apply(lambda valor: "C" if valor > 0 else "D")
                    df = df[["DATA", "DESCRIÇÃO", "VALOR", "TIPO"]]
                    df["VALOR"] = df["VALOR"].abs()

                elif "CAIXA" in extrato:
                    df.columns = df.iloc[4]
                    df = df[5:].reset_index(drop=True)

                    df["DESCRIÇÃO"] = df.apply(
                        lambda x: f'{x["HISTÓRICO"]} {x["OBS"]} {x["DADOS BANCÁRIOS"]} {x["TIPO"]} {x["Nº DOC"]}',
                        axis=1
                    )

                    df["DESCRIÇÃO"] = df["DESCRIÇÃO"].str.replace("nan", "", regex=False).str.upper()
                    df = df[["DATA", "DESCRIÇÃO", "ENTRADA", "SAIDA", "SALDO"]]

                    df["ENTRADA"] = pd.to_numeric(df["ENTRADA"], errors="coerce").fillna(0)
                    df["SAIDA"] = pd.to_numeric(df["SAIDA"], errors="coerce").fillna(0)

                    df["VALOR"] = df["ENTRADA"].where(df["ENTRADA"] != 0, df["SAIDA"] * -1)
                    df = df.loc[~df["DESCRIÇÃO"].astype(str).str.upper().str.contains("SALDO", na=False)]

                    df["TIPO"] = df["VALOR"].apply(lambda valor: "C" if valor > 0 else "D")
                    df = df[["DATA", "DESCRIÇÃO", "VALOR", "TIPO"]]
                    df["VALOR"] = df["VALOR"].abs()

                    df["DATA"] = pd.to_datetime(
                        df["DATA"],
                        errors="coerce"
                    ).dt.strftime("%d/%m/%Y")

                else:
                    continue

                arquivo_lanc = destino_lanc / f"[LANC] {arquivo.stem}_{extrato}.xlsm"

                shutil.copy2(origem_lanc, arquivo_lanc)

                planilha_lancamento(df, arquivo_lanc)

                df_totalizado = totalizador(df)
                df_totalizado.to_excel(
                    writer,
                    sheet_name=extrato,
                    index=False
                )


                display(f"Processado: {arquivo.stem} - {extrato}")

Arquivo preenchido com sucesso: G:\.shortcut-targets-by-id\1nboB01tRteAh3IDImIuSVc8ksH6cRrEH\PROJ EXT\156_CEMAF OPE\Extratos\26\04\[LANC] 0426_FECFIN_CEMAF OPE_UNICRED 10378-0.xlsm


'Processado: 0426_FECFIN_CEMAF OPE - UNICRED 10378-0'

Arquivo preenchido com sucesso: G:\.shortcut-targets-by-id\1nboB01tRteAh3IDImIuSVc8ksH6cRrEH\PROJ EXT\156_CEMAF OPE\Extratos\26\04\[LANC] 0426_FECFIN_CEMAF OPE_SICOOB 18723-2.xlsm


'Processado: 0426_FECFIN_CEMAF OPE - SICOOB 18723-2'

Arquivo preenchido com sucesso: G:\.shortcut-targets-by-id\1nboB01tRteAh3IDImIuSVc8ksH6cRrEH\PROJ EXT\156_CEMAF OPE\Extratos\26\04\[LANC] 0426_FECFIN_CEMAF OPE_SICREDI.xlsm


'Processado: 0426_FECFIN_CEMAF OPE - SICREDI'

Arquivo preenchido com sucesso: G:\.shortcut-targets-by-id\1nboB01tRteAh3IDImIuSVc8ksH6cRrEH\PROJ EXT\156_CEMAF OPE\Extratos\26\04\[LANC] 0426_FECFIN_CEMAF OPE_CAIXA.xlsm


'Processado: 0426_FECFIN_CEMAF OPE - CAIXA'